In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
from logging import INFO, WARNING, ERROR, StreamHandler, getLogger

logger = getLogger()
if not logger.hasHandlers():
    logger.addHandler(StreamHandler(sys.stdout))
logger.setLevel(INFO)

# Import

In [ ]:
import copy
import math
import os
import pathlib
from itertools import product
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm

from src.simulation.seasonal_sro import (
    SeasonalSROConfig,
    simulate_SeasonalSRO_and_write_out_results,
)
from src.utils.random_seed_helper import set_all_seeds

# Define constants

In [ ]:
ROOT_DIR = str((pathlib.Path(os.environ["PYTHONPATH"].split(":")[0]) / "..").resolve())
DATA_KIND = "KA21"  # "KA21", "V25", "H26"
assert DATA_KIND in ["KA21", "V25", "H26"]

In [ ]:
DEFAULT = SeasonalSROConfig(value_set_name=DATA_KIND)

In [ ]:
RESULT_DIR = f"{ROOT_DIR}/data/models"
os.makedirs(RESULT_DIR, exist_ok=True)

# Dependence on Ra

In [ ]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
torch.set_num_threads(1)
multiprocessing.set_start_method("spawn", force=True)
max_workers = 26 if int(os.cpu_count()) > 26 else os.cpu_count()
if DATA_KIND == "H26":
    max_workers = 61 if int(os.cpu_count()) > 61 else os.cpu_count()

print(f"{max_workers=}")

In [ ]:
lst_amp_Ra = np.linspace(0, 2.5, 26)

if DATA_KIND == "H26":
    lst_amp_Ra = np.linspace(0, 6.0, 61)

print(f"{lst_amp_Ra=}")

# Set t_max = 50_000 to reproduce our results.
for t_max, is_random_init in product([10_000], [False]):

    out_dir = f"{ROOT_DIR}/data/models/SRO_long_run_{t_max:06}yr_{DATA_KIND}"
    if is_random_init:
        out_dir += "_random_init"
    os.makedirs(out_dir, exist_ok=True)
    print(f"Calculating: {t_max=}, {is_random_init=}")

    base_seed = 42
    n_batches = 1
    t_min = 0
    n_per_month = 30  # days per month
    n_timesteps = int((t_max - t_min) * 12 * n_per_month)

    set_all_seeds(base_seed)
    x0 = torch.tensor([0.0, 10.0], dtype=torch.float64).unsqueeze(0)  # shape (1, 2)
    x0 = x0.repeat(n_batches, 1)  # shape (n_batches, 2)

    if is_random_init:
        print("Using random initial conditions.")
        x0 = torch.randn_like(x0)  # use this if you want random initial conditions

    tasks = list(enumerate(lst_amp_Ra))  # (index, amp_Ra)

    results = []
    with ProcessPoolExecutor(
        max_workers=max_workers,
    ) as ex:
        futures = []

        for index, amp in tasks:
            out_file_path = f"{out_dir}/SRO_Ra{str(amp).replace('.', 'p')[:3]}.pickle"

            fut = ex.submit(
                simulate_SeasonalSRO_and_write_out_results,
                amp,
                base_seed,
                copy.deepcopy(DEFAULT),
                x0,
                t_min,
                t_max,
                n_timesteps,
                n_batches,
                out_file_path,
                "float32",
                index,
            )
            futures.append(fut)

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Simulating"):
            try:
                res = fut.result()
                assert res["status"] == "ok" or res["status"] == "skipped-exists"
            except Exception as e:
                res = {"ERROR": repr(e)}
            results.append(res)

# Fixed Ra with R0, epsilon, or F2 varied

In [ ]:
NT_YEAR = 360
T_MAX_SDE = 50_000

# Factors 2**(k/8), k = -16..16 (0.25 to 4); R0 shifts -2.4 to +0.8 /yr in steps of 0.1 /yr.
# factor 1 is the base case
FACTORS = [2.0 ** (k / 8) for k in range(-16, 17) if k != 0]
# shift 0 is the base case
R0_SHIFTS = [round(0.1 * i, 1) for i in range(-24, 9) if i != 0]

# H26 factors left out: overdamped (no BWJ frequency) or a one-year Floquet multiplier
# >= 0.9, for which the mean of the 200-yr moment equations does not decay below 1e-7.
EXCLUDED_FACTORS = {
    ("H26", "epsilon"): [2.0 ** (k / 8) for k in [*range(-16, -11), *range(9, 17)]],
    ("H26", "F2"): [2.0 ** (k / 8) for k in (-16, -15, 14)],
}
# Wider epsilon sweep for KA21 and V25 only, 2**(k/4) = 0.0625-0.21 and 4.76-9.51
# (H26 has no damped oscillatory state there).
EPS_FACTORS_EXT = [2.0 ** (k / 4) for k in [*range(-16, -8), *range(9, 14)]]


def fixed_ra_cases() -> list[dict]:
    """The 301 cases: up to 110 per parameter set, with R_a fixed at the set's value."""
    cases = []
    for kind in ["KA21", "V25", "H26"]:
        base = SeasonalSROConfig(value_set_name=kind)
        variants = [("base", 0.0, "base", copy.deepcopy(base))]
        for d in R0_SHIFTS:
            cfg = copy.deepcopy(base)
            cfg.R0 = base.R0 + d
            label = (
                f"R0{d:+.1f}".replace("+", "plus")
                .replace("-", "minus")
                .replace(".", "p")
            )
            variants.append(("R0", d, label, cfg))
        for attr in ["epsilon", "F2"]:
            factors = FACTORS
            if attr == "epsilon" and kind != "H26":
                factors = FACTORS + EPS_FACTORS_EXT
            for f in factors:
                if f in EXCLUDED_FACTORS.get((kind, attr), []):
                    continue
                cfg = copy.deepcopy(base)
                setattr(cfg, attr, getattr(base, attr) * f)
                variants.append((attr, f, f"{attr}x{f:.2f}".replace(".", "p"), cfg))
        for family, change, label, cfg in variants:
            cases.append(
                {
                    "data_kind": kind,
                    "family": family,
                    "change": change,
                    "label": label,
                    "config": cfg,
                    "Ra_fixed": abs(base.Ra),
                    "amp_Ra": abs(base.Ra) / abs(cfg.R0),
                    "path": f"{ROOT_DIR}/data/models/fixed_Ra_SRO_long_run_{T_MAX_SDE:06}yr_{kind}/SRO_{label}.pickle",
                }
            )
    assert len(cases) == 301 and len({c["path"] for c in cases}) == len(cases)
    for c in cases:
        cfg = c["config"]
        assert cfg.R0 < cfg.epsilon  # damped
        assert cfg.F1 * cfg.F2 > (cfg.R0 + cfg.epsilon) ** 2 / 4  # oscillatory
    return cases

In [ ]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
torch.set_num_threads(1)
multiprocessing.set_start_method("spawn", force=True)
max_workers = min(128, os.cpu_count())

BASE_SEED = 2026
n_timesteps = int(T_MAX_SDE * NT_YEAR)
x0 = torch.tensor([0.0, 10.0], dtype=torch.float64).unsqueeze(0)  # shape (1, 2)

with ProcessPoolExecutor(max_workers=max_workers) as ex:
    futures = [
        ex.submit(
            simulate_SeasonalSRO_and_write_out_results,
            c["amp_Ra"],
            BASE_SEED,
            copy.deepcopy(c["config"]),
            x0,
            0,
            T_MAX_SDE,
            n_timesteps,
            1,
            c["path"],
            "float32",
            index,
        )
        for index, c in enumerate(fixed_ra_cases())
    ]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Simulating"):
        assert fut.result()["status"] in ("ok", "skipped-exists")